# Machine Learning Fundamentals: Regression with scikit-learn (Diabetes Dataset)

In this notebook, we will explore regression models using Python's **scikit-learn** library. Regression allows us to predict continuous outcomes (such as prices or measurements) based on input features.
We will start with linear regression and gradually explore non-linear models, model evaluation, overfitting, and regularization.

This notebook is designed as a **guided tutorial**. Most code is provided so you can focus on understanding the concepts rather than writing a lot of Python code.


## 1. Load and explore the dataset

We will use the **diabetes** dataset, which is included in scikit-learn.
This dataset contains ten clinical variables and measurements from n=442 diabetes patients, as well as a quantitative measure of disease progression one year after baseline. The goal is to predict the disease progression of these patients based on the baseline measurements.

You can find more information about the contents of the dataset on the [documentation page](https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset).

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

- How many observations doe we have compared to the number of features? 
- Is this a "small data" or "large data" problem?
</div>

In [ ]:
import pandas as pd
from sklearn.datasets import load_diabetes

# Load diabetes dataset and create DataFrame
_diabetes = load_diabetes()
X = _diabetes.data
y = _diabetes.target
feature_names = _diabetes.feature_names

print(f"Features shape: {X.shape}  |  Target shape: {y.shape}")
print("Features:", feature_names)

# Put into a DataFrame for convenience
X_df = pd.DataFrame(X, columns=feature_names)
df = X_df.copy()
df["target"] = y

df.head()

Now let's explore the contents of the dataset. The [*describe*](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) function is a handy tool to quickly calculate the percentiles your features.

With [*hist*](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.hist.html) we can visualize the histograms of all the features in our dataset.

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

Many regression models assume that features are on comparable scales. In our case the measurements come from a wide range of different sources (clinical parameters, blood measurements etc.). Do you think it is necessary to apply feature scaling here?
</div>


In [ ]:
from matplotlib import pyplot as plt

# Visualize feature distributions
X_df.hist(bins=15, grid=False)
plt.tight_layout()
plt.show()

# Calculate basic statistics
X_df.describe()

Now that we have a good overview of the featue value distributions, the last thing to check is the target variable. Use the following code to plot the distribution of the target variable and the correlation with the features.

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

- Look at the histogram of the target variable. Do you think this is a good distribution for a regression problem? Are there important outliers?
- What can you conclude from the feature-target correlations? Is this a good indication to try (non-)linear regression?
</div>

In [ ]:
import numpy as np

# Plot target distribution and feature-target correlations
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y, bins=20, color="#4C72B0", edgecolor="white")
axes[0].set_title("Target distribution")
axes[0].set_xlabel("Disease progression (target)")
axes[0].set_ylabel("Count")

corr = df.corr(numeric_only=True)["target"].drop("target").sort_values(key=np.abs, ascending=False)
axes[1].barh(corr.index[::-1], corr.values[::-1])
axes[1].set_title("Feature-target correlations")
axes[1].set_xlabel("Pearson r")
plt.tight_layout()
plt.show()


## 2. Baseline Linear Regression

We'll start simple by fitting a linear regression model to the dataset.

First we have to split our data in a training and test set, using scikit-learn's [*train_test_split*](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function.

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

- How would you interpret a positive vs negative coefficient?
- Can we interpret the absolute size of coefficients directly? (Hint: think about the scale of each feature (from the *.describe()* output))
</div>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit baseline Linear Regression
linreg = LinearRegression()
linreg.fit(X_train, y_train)

print("Intercept:", linreg.intercept_)
coefs = pd.Series(linreg.coef_, index=feature_names).sort_values(key=np.abs, ascending=False)
print("Top coefficients by magnitude:\n", coefs.head(5))


We can plot the fitted regression model for one feature to see if the model matches the patterns in our data.

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

- Does the relationship look strongly linear?
- Would you feel confident using this single feature to make predictions?
- What does the spread around the regression line tell you?
</div>

In [ ]:
feature_to_plot = "bmi"
feature_index = feature_names.index(feature_to_plot)
plt.figure(figsize=(6, 4))
plt.scatter(X_train[:, feature_index], y_train, color="#4C72B0", alpha=0.6, label="True target")
plt.plot(X_test[:, feature_index], linreg.intercept_ + linreg.coef_[feature_index] * X_test[:, feature_index], color='red', label='Linear fit')
plt.xlabel(feature_to_plot)
plt.ylabel("Disease progression (target)")
plt.title(f"Linear Regression Model vs True Target for feature '{feature_to_plot}'")
plt.legend()
plt.show() 

Next, we can inspect the error metrics to evaluate how well our model is doing.

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

- How do you interpret the difference between the error metrics on the training and test set? Is this a reasonable difference?
- Does a single number like R2 or MAE fully describe the model quality?
</div>

In [ ]:
# Helper to compute metrics
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

# Evaluate on train and test
train_pred = linreg.predict(X_train)
test_pred = linreg.predict(X_test)

results_baseline = pd.DataFrame(
    [regression_metrics(y_train, train_pred), regression_metrics(y_test, test_pred)],
    index=["Train", "Test"]
)
results_baseline


Finally we can plot the prediction errors against the actual prediction values. These plots help us to identify whether we are making larger errors when the predictions are low/high, or whether the variation (i.e. uncertainty) of the predictions is higher in some regions.

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

Do you spot any patterns in the residuals?
</div>

In [ ]:
# Visualize predictions vs. truth and residuals (Test set)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Predicted vs True
axes[0].scatter(y_test, test_pred, color="#4C72B0", alpha=0.8)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
axes[0].set_xlabel("True target")
axes[0].set_ylabel("Predicted target")
axes[0].set_title("Predicted vs True (Test)")

# Residuals
residuals = y_test - test_pred
axes[1].scatter(test_pred, residuals, color="#55A868", alpha=0.8)
axes[1].axhline(0, color='k', linestyle='--')
axes[1].set_xlabel("Predicted target")
axes[1].set_ylabel("Residual (y - y_hat)")
axes[1].set_title("Residuals vs Predicted (Test)")

plt.tight_layout()
plt.show()


## 3. Polynomial Regression

Nonlinear relationships can be captured by adding polynomial features (squares, interactions). Because the scales of the features change with polynomial transformations, we will need to standardize the features to keep coefficients well-behaved.

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

- Why do we apply fit_transform on the training data and transform on the test data?
- Can we swap the order of the polynomial transformation and scaling? (i.e. could we do scaling first?)
</div>


In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Build polynomial features (degree=2) and scale
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
    
scaler = StandardScaler()
X_train_poly_scaled = scaler.fit_transform(X_train_poly)
X_test_poly_scaled = scaler.transform(X_test_poly)

# Fit Linear Regression on polynomial features
linreg_poly = LinearRegression()
linreg_poly.fit(X_train_poly_scaled, y_train)

train_pred_poly = linreg_poly.predict(X_train_poly_scaled)
test_pred_poly = linreg_poly.predict(X_test_poly_scaled)

results_poly = pd.DataFrame(
    [regression_metrics(y_train, train_pred_poly), regression_metrics(y_test, test_pred_poly)],
    index=["Train (poly)", "Test (poly)"]
)

# Compare to baseline
pd.concat([results_baseline, results_poly], axis=0)


Now we can change the degree of the polynomial transformations and investigate its effect on the performance of the regression model.

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

- Which model degree looks most reasonable?
- At what point does the model start reacting to noise?
- How would you justify your choice *without* looking at test data? (Hint: how do you validate your model parameters without consuming your test set?)
</div>

In [ ]:
# Compare different polynomial degrees
results_degree = [] 
for deg in range(1, 10):
    # Build polynomial features
    poly = PolynomialFeatures(degree=deg, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    scaler = StandardScaler()
    X_train_poly_scaled = scaler.fit_transform(X_train_poly)
    X_test_poly_scaled = scaler.transform(X_test_poly)
    
    # Fit Linear Regression
    linreg_poly = LinearRegression()
    linreg_poly.fit(X_train_poly_scaled, y_train)
    
    # Predict
    train_pred_poly = linreg_poly.predict(X_train_poly_scaled)
    test_pred_poly = linreg_poly.predict(X_test_poly_scaled)
    
    # Store results
    train_metrics = regression_metrics(y_train, train_pred_poly)
    test_metrics = regression_metrics(y_test, test_pred_poly)
    results_degree.append({
        "Degree": deg,
        "Train_MAE": train_metrics["MAE"],
        "Test_MAE": test_metrics["MAE"]
    })

# Plot the results
results_degree_df = pd.DataFrame(results_degree)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(results_degree_df["Degree"], results_degree_df["Train_MAE"], marker='o', label="Train MAE")
ax.plot(results_degree_df["Degree"], results_degree_df["Test_MAE"], marker='o', label="Test MAE")
ax.set_xlabel("Polynomial Degree")
ax.set_ylabel("MAE")
ax.set_title("MAE vs Polynomial Degree")
ax.legend()
plt.tight_layout()

## 4. Regularization to control complexity

Finally we can include regularization to control the complexity of the fitted models. Polynomial features increase model flexibility and can allow it to detect more detailed patterns, but also increases the risk of overfitting. Regularization (Ridge, Lasso, ElasticNet) shrinks coefficients to compensate for this effect and improve generalization.

Use the code below to fit a non-regularized and regularized polynomial regression model. Look at the effect of regularization on the magnitude of the model coefficients.

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

- How does regularization affect the coefficient values?
- Are there differences between the effect of Ridge or Lasso on the coefficient values?
- Which model (Ridge or Lasso) would you prefer if interpretability matters?
</div>


In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

# Build features
poly = PolynomialFeatures(2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_train_poly = StandardScaler().fit_transform(X_train_poly)

# Fit models
linreg_poly = LinearRegression()
ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.1)
elasticnet = ElasticNet(alpha=0.1, l1_ratio=0.5)

linreg_poly.fit(X_train_poly, y_train)
ridge.fit(X_train_poly, y_train)
lasso.fit(X_train_poly, y_train)
elasticnet.fit(X_train_poly, y_train)

# Plot coefficient magnitudes
fig, ax = plt.subplots(figsize=(9, 4))

coef_lin = np.abs(linreg_poly.coef_)
coef_reg = np.abs(ridge.coef_)
coef_lasso = np.abs(lasso.coef_)
coef_enet = np.abs(elasticnet.coef_)

x = np.arange(len(coef_lin))
width = 0.2

ax.bar(x - width*1.5, coef_lin, width=width, label="Unregularized")
ax.bar(x - width*0.5, coef_reg, width=width, label="Ridge Regularized")
ax.bar(x + width*0.5, coef_lasso, width=width, label="Lasso Regularized")
ax.bar(x + width*1.5, coef_enet, width=width, label="ElasticNet Regularized")

ax.set(
    xlabel="Feature Index",
    ylabel="|Coefficient|",
    title=f"Coefficient Magnitudes: Unregularized vs Regularized",
    xlim=(-1, 11),
    ylim=(0,100)
)
ax.legend()
plt.show()



Now that we know how regularization works, we can inspect how it affects the generalization of the fitted model. To do this, we compute one of the error metrics (MAE) for each type of regularization across a range of polynomial degrees.

>Warning: this code can take a while to run depending on your hardware, which is why we limit the range of polynomial degrees between 1-5.

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

- Why might regularization improve test performance but worsen training performance?
- Which regularization model gives the best generalization results?
</div>

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

models = {
    "Linear (poly)": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "Lasso": Lasso(alpha=0.1, max_iter=10000, random_state=42),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000, random_state=42),
}

results = []

# Loop over polynomial degrees
for deg in range(1, 5):
    print(f"Processing polynomial degree: {deg}")
    poly = PolynomialFeatures(degree=deg, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    scaler = StandardScaler()
    X_train_poly_scaled = scaler.fit_transform(X_train_poly)
    X_test_poly_scaled = scaler.transform(X_test_poly)
    
    for model_name, model in models.items():
        # Fit model
        model.fit(X_train_poly_scaled, y_train)

        # Predict
        train_pred = model.predict(X_train_poly_scaled)
        test_pred = model.predict(X_test_poly_scaled)

        # Store results
        train_metrics = regression_metrics(y_train, train_pred)
        test_metrics = regression_metrics(y_test, test_pred)
        results.append({
            "Model": model_name,
            "Degree": deg,
            "Train_MAE": train_metrics["MAE"],
            "Test_MAE": test_metrics["MAE"]
        })

# Plot the results
results_df = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(6, 4))

for model_name in models.keys():
    subset = results_df[results_df["Model"] == model_name]
    ax.plot(
        subset["Degree"],
        subset["Test_MAE"],
        marker="o",
        label=f"{model_name} (Test)"
    )

ax.set_xlabel("Polynomial Degree")
ax.set_ylabel("MAE")
ax.set_title("Test MAE vs Polynomial Degree for Different Regularization Methods")
ax.legend()
plt.show()


## 5. Wrap-up

In this notebook, we explored the progression from simple to more flexible regression models:

- **Linear regression** serves as a strong, interpretable baseline.
- **Evaluation metrics** such as MAE, RMSE, and R² allow us to compare training and test performance and diagnose overfitting.
- **Residual analysis** helps uncover systematic patterns that signal model misspecification.
- **Polynomial features** increase model flexibility, while feature standardization improves numerical stability and optimization.
- **Regularization techniques** (Ridge, Lasso, and ElasticNet) control model complexity by shrinking coefficients, often leading to better generalization.

Together, these tools from a practical workflow for building, diagnosing, and improving regression models in real-world settings.

---

## Optional exercises (for early finishers)

These sections are optional. You can run them to deepen your understanding.


### A. Collinearity checks

Highly correlated features can lead to unstable coefficient estimates and make model interpretation difficult. In this exercise, we will:
- Visualize the pairwise correlations between the original features using a correlation heatmap
- Compute the Variance Inflation Factor (VIF) to quantify multicollinearity

These diagnostics help identify redundancy in the feature set and inform whether feature selection or regularization may be beneficial.

<blockquote>
<b>Interpretation:</b>

- Highly correlated features can make coefficients unstable and harder to interpret. Predictions can still be good, but coefficient signs and magnitudes may fluctuate. In the heatmap, look for correlation values near ±0.7–0.8; these indicate potential redundancy.
- The Variance Inflation Factor (VIF) models the same effect by qantifying how much the variance of a coefficient is inflated due to correlations with other features. Values above 5–10 suggest problematic multicollinearity.
</blockquote>

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

- What can you say about the collinearity in this dataset?
- Is it important to address the collinearity if the main goal is prediction?
- Can you think of a way to reduce the collinearity in the dataset?
</div>

In [ ]:
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Correlation heatmap (original features)
plt.figure(figsize=(7,6))
sns.heatmap(X_df.corr(numeric_only=True), annot=False, cmap="vlag", center=0)
plt.title("Feature correlation heatmap (original X)")
plt.tight_layout()
plt.show()

# VIF with statsmodels
X_const = sm.add_constant(X_df)  # add intercept for VIF computation
vif = pd.DataFrame()
vif["feature"] = X_const.columns
vif["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
vif = vif[vif["feature"] != "const"].sort_values("VIF", ascending=False)
display(vif)


## B. Dimensionality reduction

Highly correlated features can inflate coefficient variance and make interpretation difficult. One way to reduce this effect is to project the data into a lower-dimensional space using [Principal Component Analysis (PCA)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html).

<blockquote>
PCA is an unsupervised technique that can be used to reduce the dimensionality of your dataset.<br>
Each PCA component is a linear combination of the original features, and is computed in such a way that all the components are orthogonal (i.e. uncorrelated) to each other.<br>
The components are also ranked by how much of the variance of the dataset they explain.<br>
In other words, the first component contains the most variance of our dataset, the second component is orthogonal to the first and contains slightly less variance, and so on.

By keeping only the first *N* components, we obtain a lower-dimensional feature space (with dimensionality *N*) with features that are linearly independent of each other.

If you want to learn more about PCA, you can check <a href url="https://builtin.com/data-science/step-step-explanation-principal-component-analysis">this step by step explanation </a>.
</blockquote>

<div class="alert alert-block alert-info"> 
<b>QUESTIONS</b>

- Look at the cumulative explained variance curve. After how many components does it platteau?
- What does this tell you about the amount of redundant features?
- If we fit a regression model on the PCA components, what will be the effect on the interpretability?
</div>

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
X_train_pca = pca.fit_transform(X_train)  # Use the polynomial features from earlier
X_test_pca = pca.transform(X_test)

plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.show()